# Build a Knowledge Graph for LLM Training

End-to-end example: **download a small corpus → build a knowledge graph → export it → generate QA training pairs** for fine-tuning an LLM.

Everything uses Polygraph's public API, so you can swap in your own documents (JSONL with `id` + `text` fields, see `docs/input_data_format.md`) at any point.

In [ ]:
from polygraph.data import Data, RandomSampler
from polygraph.pipelines import Baseline

# 1. Fetch a small Wikipedia sample (automatically enriched with hyperlinks)
Data.download(
    "wikipedia",
    path="data/llm_demo/articles.jsonl",
    sampler=RandomSampler(count=20),
)

In [ ]:
# 2. Build the knowledge graph: preprocess → extract → resolve → build
pipe = Baseline(
    input_paths=["data/llm_demo/"],
    output_dir="output/llm_demo/",
)

chunks = pipe.preprocess()
kg = pipe.build_kg(chunks)
pipe.export(kg)

# The built KG is also exposed as an in-memory GraphStore — the same read
# interface whether the graph lives in RAM, a file, or Neo4j.
store = kg["graph_store"]  # identical to pipe.graph_store
print(f"KG: {store.number_of_nodes()} nodes, {store.number_of_edges()} edges")
print(f"Exported → {pipe.output_dir}/knowledge_graph.json")

## Use the KG for LLM training

The exported knowledge graph (`knowledge_graph.json`) can be used to generate
QA pairs for **supervised fine-tuning (SFT)** — for example, single-hop
questions from entity–relation triples, or multi-hop questions that traverse
the graph.